![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

# 1) Read data
df = pd.read_csv("rental_info.csv")

# 2) Create rental_length_days
df["rental_date"] = pd.to_datetime(df["rental_date"])
df["return_date"] = pd.to_datetime(df["return_date"])
df["rental_length_days"] = (df["return_date"] - df["rental_date"]).dt.days

# 3) Dummy variables from special_features
special = df["special_features"].fillna("").astype(str)
df["deleted_scenes"] = special.str.contains("Deleted Scenes", regex=False).astype(int)
df["behind_the_scenes"] = special.str.contains("Behind the Scenes", regex=False).astype(int)

# 4) Feature matrix X (avoid leakage)
leaky_cols = {"rental_length_days", "rental_date", "return_date", "special_features"}
feature_cols = [c for c in df.columns if c not in leaky_cols]
X = df[feature_cols].copy()

# Basic safety: if any missing values exist, fill with 0 for modeling
X = X.fillna(0)

# 5) Target y
y = df["rental_length_days"].copy()

# Train/test split (20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=9
 )

# Try multiple regression models and pick the best by test MSE
candidates = {
    "linear": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(random_state=9))]),
    "lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(random_state=9, max_iter=20000))]),
    "gbr": GradientBoostingRegressor(random_state=9),
    "rf": RandomForestRegressor(random_state=9, n_estimators=400, n_jobs=-1),
    "etr": ExtraTreesRegressor(random_state=9, n_estimators=600, n_jobs=-1),
}

best_name = None
best_model = None
best_mse = float("inf")

for name, model in candidates.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    if mse < best_mse:
        best_mse = mse
        best_model = model
        best_name = name

best_name, best_mse

('rf', 2.0259952866020243)